# Certifying fault tolerance

This notebook shows how to compute exact certificates of fault tolerance for noisy Stim circuits.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim
import sympy

from qldpc import circuits, codes

## Certify the fault distance of a memory experiment

The fault distance of a noisy circuit is the smallest number of faults that flip a logical observable without firing any detector.
Enumerating every combination of faults is impractical beyond small circuits: the bivariate bicycle example below has 432 faults after merging equivalent error mechanisms, so checking every combination of up to 6 of them would take $\sum_{k=1}^{6} \binom{432}{k} \approx 9 \times 10^{12}$ steps.
`circuits.get_fault_distance` instead converts the circuit into an auxiliary CSS code and uses the Brouwer-Zimmermann code-distance algorithm, which stops as soon as it can prove that no lighter undetected fault exists.
`circuits.get_fault_distance` thereby computes the fault distance exactly in a few seconds here, and it scales to much larger circuits, though its cost is still exponential in the fault distance.
See `help(circuits.get_fault_distance)` for additional information.

Here we compute the fault distance of X- and Z-basis memory experiments for the [[72,12,6]] bivariate bicycle code from [arXiv:2308.07915](https://arxiv.org/abs/2308.07915), with one round of syndrome extraction under circuit-level depolarizing noise.

In [3]:
x, y = sympy.symbols("x y")
code = codes.BBCode({x: 6, y: 6}, x**3 + y + y**2, y**3 + x + x**2)
noise_model = circuits.DepolarizingNoiseModel(1e-3)


def get_noisy_memory_circuit(code, basis, num_rounds=1):
    # one logical memory cycle = num_rounds rounds of syndrome extraction
    # you can replace logical_memory_cycle with another memory circuit
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=num_rounds
    )

    # keep initialization and readout noiseless, to certify only the inner QEC cycle
    return initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout


print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)

    # the number of detectors flipped by each fault of the detector error model
    detectors_per_fault = [
        sum(target.is_relative_detector_id() for target in instruction.targets_copy())
        for instruction in noisy_circuit.detector_error_model().flattened()
        if instruction.type == "error"
    ]
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    print(
        f"{basis}-basis memory: {len(detectors_per_fault)} faults, "
        f"each flipping up to {max(detectors_per_fault)} detectors, "
        f"fault distance {fault_distance}"
    )

code parameters: (72, 12, 6)
X-basis memory: 432 faults, each flipping up to 6 detectors, fault distance 6
Z-basis memory: 432 faults, each flipping up to 6 detectors, fault distance 6


### The Steane code

The above example uses bare-ancilla syndrome extraction.
However, this syndrome measurement strategy does not always preserve the distance of a code.
In the case of the Steane code, the fault distance of the bare-ancilla syndrome measurement strategy is less than the distance of the Steane code, indicating that other (e.g., flagged, Shor, or Steane-type QEC) strategies are necessary for fault-tolerant QEC.

In [4]:
code = codes.SteaneCode()

print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)
    print(f"{basis}-basis memory: fault distance {circuits.get_fault_distance(noisy_circuit)}")

code parameters: (7, 1, 3)
X-basis memory: fault distance 2
Z-basis memory: fault distance 2


## Certify fault-tolerant state preparation

`circuits.certify_bounded_fault_propagation` checks that every set of faults leaves an output error that is equivalent, up to a stabilizer of the prepared state, to an error with weight at most the number of faults.
Bounded fault propagation is a sufficient criterion for fault-tolerant state preparation; see Definition 3 in Section 4.2 of [arXiv:0904.2557](https://arxiv.org/abs/0904.2557).
See `help(circuits.certify_bounded_fault_propagation)` for additional information.

Here we prepare GHZ states by fanning out CX gates, and add circuit-level depolarizing noise with a noise model.

In [5]:
noise_model = circuits.DepolarizingNoiseModel(1e-3)

for num_qubits in (3, 4):
    ghz_circuit = stim.Circuit("H 0")
    for target in range(1, num_qubits):
        ghz_circuit.append("CX", [0, target])
    noisy_circuit = noise_model.noisy_circuit(ghz_circuit)

    certificate = circuits.certify_bounded_fault_propagation(noisy_circuit)
    print(f"{num_qubits}-qubit GHZ state: certified = {certificate.certified}")
    if witness := certificate.counterexample:
        print(f"  faults {witness.fault_indices}: error equivalent to {witness.output_pauli}")
        print(f"  equivalent errors have weight at least {witness.effective_weight_lower_bound}")

3-qubit GHZ state: certified = True
4-qubit GHZ state: certified = False
  faults (4,): error equivalent to +ZXX_
  equivalent errors have weight at least 2


With four qubits, one fault already leaves an error that no stabilizer reduces to weight one, so the certificate fails and returns that fault as a counterexample.
The printed error is one member of its equivalence class, not necessarily the lightest.

### Post-selection

Repeat-until-success preparation protocols discard the attempts in which a flag detector fires, so only the sets of faults that fire no flag must meet the bound.
Here we flag the preparation of the Steane code's logical zero state: `add_flags=True` adds a detector for each measurement that no detector addresses, and `post_select="flags"` post-selects on them.

In [6]:
prep_circuit = stim.Circuit("""
    # non-fault-tolerant preparation of the logical zero state
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag qubit 7 detects errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")
noisy_circuit = circuits.DepolarizingNoiseModel(1e-3).noisy_circuit(prep_circuit)

for post_select, css_weight in [(False, None), ("flags", None), ("flags", False)]:
    certificate = circuits.certify_bounded_fault_propagation(
        noisy_circuit, add_flags=True, post_select=post_select, css_weight=css_weight
    )
    print(
        f"post_select={post_select!r}, CSS weight {certificate.css_weight}: "
        f"certified = {certificate.certified}"
    )

post_select=False, CSS weight True: certified = False
post_select='flags', CSS weight True: certified = True
post_select='flags', CSS weight False: certified = False


Post-selecting on the flag certifies the preparation under CSS weight but not physical weight, which counts an error with X on one qubit and Z on another as weight two.

### Sets of several faults

Two faults can pass post-selection together by cancelling each other's flag.
The default `max_faults=None` allows sets of any number of faults, and an integer limits the number of faults per set.

In [7]:
shared_flag_circuit = stim.Circuit("""
    I 0 1 2 3 4
    CORRELATED_ERROR(0.01) X0 X1 X4
    CORRELATED_ERROR(0.01) X2 X3 X4
    M 4
""")

for max_faults in (1, None):
    certificate = circuits.certify_bounded_fault_propagation(
        shared_flag_circuit, max_faults=max_faults, add_flags=True, post_select="flags"
    )
    print(f"max_faults={max_faults}: certified = {certificate.certified}")
    if witness := certificate.counterexample:
        print(f"  faults {witness.fault_indices}: error equivalent to {witness.output_pauli}")
        print(f"  equivalent errors have weight at least {witness.effective_weight_lower_bound}")

max_faults=1: certified = True
max_faults=None: certified = False
  faults (0, 1): error equivalent to +XXXX
  equivalent errors have weight at least 4


Each fault alone is discarded, but the pair passes and leaves an error of weight four, more than its two faults.